# Programowanie w języku Python

## Wykład 15 — Od skryptu do aplikacji: integracja kursu

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozumieć, jak połączyć materiał całego kursu w jeden projekt,
- umieć rozdzielić kod na moduły i odpowiedzialności,
- korzystać z konfiguracji, plików i zmiennych środowiskowych,
- zbudować prosty interfejs wiersza poleceń,
- stosować klasy tylko tam, gdzie dają realną korzyść,
- używać logowania i wyjątków w punkcie wejścia programu,
- serializować wyniki,
- rozumieć miejsce testów w projekcie,
- potrafić przejść od problemu do struktury aplikacji.

## Wracamy do pytania z W1

Na początku kursu mieliśmy schemat:

PROBLEM → ALGORYTM → PROGRAM → WYNIK

Dzisiaj przechodzimy przez ten sam proces dla programu, który ma wiele plików, konfigurację, testy i trwałe wyniki.

## Problem końcowy

Zbudujemy małą aplikację:

> wczytaj pomiary czujników, sprawdź ich poprawność, wykryj alarmy i zapisz raport.

Przykładowe wejście:

sensor_01;temperature;21.7;C
sensor_02;temperature;35.2;C
sensor_03;temperature;-300;C

## Wymagania

Program ma:

1. przyjąć ścieżkę do pliku wejściowego,
2. odczytać rekordy,
3. odrzucić niepoprawne pomiary,
4. oznaczyć alarmy,
5. zapisać raport JSON,
6. logować przebieg,
7. zwracać sensowny kod zakończenia,
8. być możliwy do przetestowania.

## Najpierw dzielimy odpowiedzialności

CLI → konfiguracja

PARSER → tekst na dane

MODEL / REGUŁY → walidacja i alarm

RAPORT → serializacja JSON

MAIN → koordynacja

TESTY → weryfikacja zachowania

## Proponowana struktura projektu

sensor_app/
├── pyproject.toml
├── src/
│   └── sensor_app/
│       ├── __init__.py
│       ├── cli.py
│       ├── parser.py
│       ├── models.py
│       ├── rules.py
│       ├── report.py
│       └── main.py
└── tests/
    ├── test_parser.py
    └── test_rules.py

## Model danych

Dla pojedynczego pomiaru dobrze pasuje dataclass:

In [ ]:
from dataclasses import dataclass


@dataclass
class Measurement:
    sensor: str
    kind: str
    value: float
    unit: str

## Dlaczego dataclass?

Obiekt ma głównie przechowywać dane, być czytelnie reprezentowany i mieć prostą semantykę równości.

Nie potrzebujemy rozbudowanej hierarchii klas.

## Parser

Parser powinien robić jedno:

tekst → Measurement

In [ ]:
def parse_measurement(line: str) -> Measurement:
    sensor, kind, value, unit = [
        part.strip()
        for part in line.split(';')
    ]

    return Measurement(
        sensor=sensor,
        kind=kind,
        value=float(value),
        unit=unit,
    )

## Własny wyjątek domenowy

Możemy oddzielić błąd formatu wejściowego od szczegółów implementacji.

In [ ]:
class MeasurementParseError(Exception):
    """Invalid measurement input record."""


def parse_measurement_safe(line: str) -> Measurement:
    try:
        return parse_measurement(line)
    except (ValueError, TypeError) as error:
        raise MeasurementParseError(
            f"Niepoprawny rekord: {line!r}"
        ) from error

## Reguła poprawności

Reguły domenowe nie muszą wiedzieć, czy dane przyszły z pliku, API czy klawiatury.

In [ ]:
def is_valid(measurement: Measurement) -> bool:
    if measurement.kind == 'temperature':
        return measurement.value >= -273.15

    return True

## Reguła alarmu

Limit jest parametrem, a nie ukrytą globalną stałą.

In [ ]:
def is_alarm(
    measurement: Measurement,
    temperature_limit: float = 30.0,
) -> bool:
    return (
        measurement.kind == 'temperature'
        and measurement.value > temperature_limit
    )

## Konfiguracja

Dobre rozdzielenie:

zmienna środowiskowa / CLI / plik konfiguracyjny
→ main()
→ konkretna wartość
→ funkcja domenowa

Warstwa reguł nie musi wiedzieć, skąd pochodzi konfiguracja.

In [ ]:
import os

alarm_limit = float(
    os.getenv('SENSOR_ALARM_LIMIT', '30')
)

print(alarm_limit)

## Interfejs CLI

Do prostego interfejsu wiersza poleceń użyjemy argparse.

In [ ]:
import argparse


def build_parser():
    parser = argparse.ArgumentParser(
        description='Analyze sensor measurements'
    )

    parser.add_argument('input')
    parser.add_argument('--output', default='report.json')
    parser.add_argument('--limit', type=float, default=30.0)

    return parser

## Argumenty programu

Przykładowe uruchomienie:

python -m sensor_app.main data/input.txt

lub:

python -m sensor_app.main data/input.txt --limit 35

## Odczyt pliku

Wracamy do pathlib i generatorów.

In [ ]:
from pathlib import Path


def read_lines(path: str | Path):
    path = Path(path)

    with path.open('r', encoding='utf-8') as file:
        yield from file

## Dlaczego generator?

read_lines() zwraca kolejne linie na żądanie.

Nie musi ładować całego pliku do pamięci.

To praktyczne zastosowanie materiału z W12.

## Pipeline danych

plik
→ generator linii
→ parser
→ Measurement
→ walidacja
→ reguła alarmu
→ raport

In [ ]:
def analyze_lines(lines, temperature_limit=30.0):
    valid = []
    invalid = []
    alarms = []

    for line in lines:
        try:
            measurement = parse_measurement_safe(line)
        except MeasurementParseError as error:
            invalid.append({
                'line': line.strip(),
                'error': str(error),
            })
            continue

        if not is_valid(measurement):
            invalid.append({
                'line': line.strip(),
                'error': 'Invalid physical value',
            })
            continue

        valid.append(measurement)

        if is_alarm(measurement, temperature_limit):
            alarms.append(measurement)

    return valid, invalid, alarms

## Wynik analizy jako dane

Jeżeli trzy osobne listy zaczynają utrudniać kod, możemy wprowadzić kolejną małą strukturę danych.

In [ ]:
@dataclass
class AnalysisResult:
    valid: list[Measurement]
    invalid: list[dict]
    alarms: list[Measurement]

## Serializacja obiektów

JSON nie zna automatycznie naszej klasy Measurement.

Musimy wskazać reprezentację prostych danych.

In [ ]:
from dataclasses import asdict

measurement = Measurement(
    'sensor_01',
    'temperature',
    21.7,
    'C',
)

print(asdict(measurement))

In [ ]:
import json


def save_report(path: str | Path, result: AnalysisResult):
    path = Path(path)

    data = {
        'valid': [asdict(item) for item in result.valid],
        'invalid': result.invalid,
        'alarms': [asdict(item) for item in result.alarms],
    }

    with path.open('w', encoding='utf-8') as file:
        json.dump(
            data,
            file,
            indent=2,
            ensure_ascii=False,
        )

## Logging

Oddzielamy wynik programu od informacji diagnostycznej.

In [ ]:
import logging

logger = logging.getLogger(__name__)


def configure_logging(verbose=False):
    level = logging.DEBUG if verbose else logging.INFO

    logging.basicConfig(
        level=level,
        format='%(levelname)s %(name)s: %(message)s',
    )

## main() jako koordynator

Rola main():

konfiguracja → uruchomienie funkcji → obsługa błędów granicznych → kod zakończenia

In [ ]:
def main(argv=None):
    parser = build_parser()
    args = parser.parse_args(argv)

    configure_logging()

    try:
        valid, invalid, alarms = analyze_lines(
            read_lines(args.input),
            temperature_limit=args.limit,
        )

        result = AnalysisResult(valid, invalid, alarms)
        save_report(args.output, result)

    except FileNotFoundError:
        logger.error('Nie znaleziono pliku: %s', args.input)
        return 1
    except PermissionError:
        logger.error('Brak uprawnień do pliku')
        return 2

    logger.info('Zapisano raport: %s', args.output)
    return 0

## Punkt wejścia

W pliku main.py możemy zakończyć:

if __name__ == '__main__':
    raise SystemExit(main())

Wartość zwrócona przez main() staje się kodem zakończenia procesu.

## Testujemy logikę bez CLI

Dzięki podziałowi projektu testujemy reguły bez uruchamiania całej aplikacji.

In [ ]:
def test_alarm_rule():
    measurement = Measurement(
        'sensor_01',
        'temperature',
        35.0,
        'C',
    )

    assert is_alarm(
        measurement,
        temperature_limit=30,
    )

In [ ]:
def test_invalid_temperature():
    measurement = Measurement(
        'sensor_01',
        'temperature',
        -300.0,
        'C',
    )

    assert not is_valid(measurement)

## Test jednostkowy a integracyjny

Test reguły sprawdza mały fragment kodu.

Test pliku może sprawdzać cały przebieg:

zapis → odczyt → parser → wynik.

Oba poziomy są potrzebne, ale odpowiadają na inne pytania.

## Gdzie współbieżność?

Dopiero po zbudowaniu poprawnej aplikacji pytamy, czy istnieje wąskie gardło.

Dla jednego lokalnego pliku prawdopodobnie nie.

Dla 100 urządzeń sieciowych ThreadPoolExecutor lub asyncio może mieć sens.

## OOP nie jest obowiązkowe wszędzie

W projekcie użyliśmy klasy dla danych, ale reguły pozostawiliśmy jako zwykłe funkcje.

To celowa decyzja.

OOP jest narzędziem do zarządzania złożonością, nie obowiązkowym stylem każdej funkcji.

## Dekoratory też są narzędziem

Moglibyśmy dodać dekorator do pomiaru czasu, logowania lub retry.

Jeśli nie mamy realnej potrzeby, nie dokładamy abstrakcji tylko dlatego, że ją poznaliśmy.

## Minimalny pyproject.toml

Przykład konfiguracji projektu:

[project]
name = "sensor-app"
version = "0.1.0"
requires-python = ">=3.12"

[project.scripts]
sensor-app = "sensor_app.main:main"

## Od notebooka do aplikacji

Notebook świetnie służy do nauki i eksperymentów.

Aplikacja potrzebuje dodatkowo:

- struktury modułów,
- punktu wejścia,
- konfiguracji,
- obsługi błędów,
- testów,
- powtarzalnego środowiska.

## Jak rozwijać program krok po kroku?

1. Rozwiąż problem dla jednego przykładu.
2. Wydziel funkcje.
3. Oddziel I/O od logiki.
4. Dodaj testy.
5. Podziel kod na moduły.
6. Dodaj konfigurację.
7. Dodaj diagnostykę.
8. Zmierz wydajność.
9. Dopiero wtedy optymalizuj lub dodawaj współbieżność.

## Czego nie robimy?

Nie projektujemy od razu:

- pięciu warstw abstrakcji,
- hierarchii 20 klas,
- własnego frameworka,
- metaklas,
- globalnego singletona konfiguracji,
- współbieżności bez pomiaru.

Złożoność powinna wynikać z problemu.

## Mapa całego kursu

W1  problem → algorytm → program
W2  dane i obiekty
W3  sterowanie
W4  funkcje I
W5  funkcje II
W6  tekst
W7  moduły i środowisko
W8  pliki i serializacja
W9  wyjątki i diagnostyka
W10 OOP I
W11 OOP II
W12 iteracja, generatory, dekoratory
W13 jakość i testy
W14 współbieżność
W15 aplikacja

## Najważniejsza lekcja

Python to nie lista składni do zapamiętania.

W ciągu kursu budowaliśmy kolejne poziomy:

wartość → obiekt → sterowanie → funkcja → moduł → pakiet → aplikacja

## Program można rozumieć na wielu poziomach

SKŁADNIA → jak to zapisać?

SEMANTYKA → co to znaczy?

MODEL OBIEKTÓW → jakie obiekty istnieją?

STRUKTURA → jak podzielić odpowiedzialności?

WYKONANIE → co dzieje się podczas działania?

JAKOŚĆ → jak sprawdzić poprawność?

WYDAJNOŚĆ → gdzie jest koszt?

ŚRODOWISKO → gdzie i z czym program działa?

## Zadanie końcowe

Rozbuduj aplikację pomiarową tak, aby:

1. posiadała strukturę pakietu,
2. przyjmowała argumenty CLI,
3. korzystała z konfiguracji środowiskowej,
4. czytała dane wejściowe z pliku,
5. używała Measurement jako dataclass,
6. walidowała dane i wykrywała alarmy,
7. zapisywała raport JSON,
8. logowała przebieg,
9. miała co najmniej 5 automatycznych testów,
10. zawierała README z instrukcją utworzenia .venv i uruchomienia programu.

## Pytania kontrolne

Po kursie powinieneś potrafić odpowiedzieć:

- kiedy użyć listy, a kiedy słownika?
- kiedy funkcja powinna zwracać wartość zamiast ją wypisywać?
- kiedy potrzebujemy klasy?
- skąd Python znajduje moduł?
- czym różni się str od bytes?
- czym różni się JSON od pickle?
- po co istnieją wyjątki?
- kiedy generator jest lepszy od listy?
- jak sprawdzić poprawność kodu?
- kiedy wątek pomoże, a kiedy nie?

## Koniec kursu

### Od prostych instrukcji do struktury kompletnej aplikacji

Najważniejsze nie jest zapamiętanie wszystkich elementów języka.

Najważniejsze jest umieć:

> rozłożyć problem, dobrać właściwe narzędzia, napisać czytelny kod i sprawdzić, czy działa poprawnie.